# Genie Code - Asistente IA para Machine Learning

## 🤖 ¿Qué es Genie Code?

**Genie Code** es el asistente de IA de Databricks que ayuda a desarrollar pipelines de ML más rápido mediante lenguaje natural.

### 🎯 Capacidades

* ✅ **Generar código**: "Create a Random Forest for classification"
* ✅ **Feature engineering**: "Add polynomial features for this dataset"
* ✅ **Debugging**: "Why is my model overfitting?"
* ✅ **Explicar código**: "Explain this PySpark transformation"
* ✅ **Optimizar**: "How can I improve model performance?"
* ✅ **Visualizaciones**: "Create a confusion matrix heatmap"
* ✅ **Documentación**: "Add docstrings to this function"

### 💡 Workflow

1. 📝 **Describir** lo que necesitas en lenguaje natural
2. 🤖 **Genie genera** código optimizado
3. ▶️ **Ejecutar** y verificar resultados
4. 🔄 **Iterar** con Genie para refinamiento

---

## 🚀 Ventajas vs Codificación Manual

| Aspecto | Manual | Con Genie |
|---------|--------|----------|
| **Tiempo** | 30-60 min | 2-5 min |
| **Sintaxis** | Buscar docs | Auto-generado |
| **Mejores prácticas** | Depende de expertise | Incorporadas |
| **Debugging** | Stack Overflow | Sugerencias instantáneas |
| **Aprendizaje** | Lento | Rápido (ver código generado) |

---

## 📖 Casos de Uso

En este notebook veremos ejemplos de:

1. Generar datasets sintéticos
2. Feature engineering avanzado
3. Entrenar modelos con mejores prácticas
4. Debugging de errores comunes
5. Optimizar performance
6. Mejores prácticas de prompts

## Ejemplo 1: Generar Dataset Sintético

### 💬 Prompt para Genie:

```
Generate a synthetic dataset for customer churn prediction with:
- 1000 rows
- Features: age, tenure, monthly_charges, contract_type, internet_service
- Target: churn (binary, ~25% churn rate)
- Make churn correlated with: short tenure, high charges, month-to-month contracts
```

### 🤖 Genie generaría algo como:

```python
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

df_churn = pd.DataFrame({
    'age': np.random.randint(18, 80, n),
    'tenure': np.random.randint(1, 72, n),
    'monthly_charges': np.random.uniform(20, 120, n),
    'contract_type': np.random.choice(['Monthly', 'Annual', 'Biennial'], n, p=[0.6, 0.3, 0.1]),
    'internet_service': np.random.choice(['DSL', 'Fiber', 'No'], n, p=[0.4, 0.4, 0.2])
})

# Generar churn con lógica de negocio
churn_prob = 0.1
churn_prob = np.where(df_churn['contract_type'] == 'Monthly', 0.4, 0.1)
churn_prob = np.where(df_churn['tenure'] < 12, churn_prob * 2, churn_prob)
churn_prob = np.where(df_churn['monthly_charges'] > 80, churn_prob * 1.5, churn_prob)
churn_prob = np.clip(churn_prob, 0, 1)
df_churn['churn'] = np.random.binomial(1, churn_prob)

print(f"Churn rate: {df_churn['churn'].mean():.1%}")
```

### ✅ Beneficio

**Ahorra 15-20 minutos** de setup manual. Genie entiende:
* Generar correlaciones realistas
* Balancear distribuciones
* Aplicar lógica de negocio al target

## Ejemplo 2: Feature Engineering Avanzado

### 💬 Prompt:

```
Create these features for the churn dataset:
1. Interaction: monthly_charges * tenure (total_spend)
2. Binning: age into groups (18-30: Young, 31-60: Adult, 60+: Senior)
3. One-hot encode contract_type and internet_service
4. Create a flag: high_value_customer (monthly_charges > 75th percentile)
```

### 🤖 Genie generaría:

```python
from sklearn.preprocessing import LabelEncoder

# 1. Interaction feature
df_churn['total_spend'] = df_churn['monthly_charges'] * df_churn['tenure']

# 2. Age binning
df_churn['age_group'] = pd.cut(
    df_churn['age'], 
    bins=[0, 30, 60, 100], 
    labels=['Young', 'Adult', 'Senior']
)

# 3. One-hot encoding
df_encoded = pd.get_dummies(
    df_churn, 
    columns=['contract_type', 'internet_service'], 
    prefix=['contract', 'internet'],
    drop_first=True  # Evitar multicolinealidad
)

# 4. High value flag
threshold = df_churn['monthly_charges'].quantile(0.75)
df_encoded['high_value_customer'] = (df_encoded['monthly_charges'] > threshold).astype(int)

print(f"Features creados: {len(df_encoded.columns)}")
print(f"High value customers: {df_encoded['high_value_customer'].sum()}")
```

### ✅ Beneficio

**Genera transformaciones comunes instantáneamente**, siguiendo mejores prácticas:
* Evita multicolinealidad con `drop_first=True`
* Usa quantiles para umbrales dinámicos
* Nombres descriptivos de columnas

## Ejemplo 3: Entrenar Modelo con Mejores Prácticas

### 💬 Prompt:

```
Train a Random Forest classifier for churn prediction:
- Train/test split 80/20 stratified
- Hyperparameters: n_estimators=100, max_depth=10
- Evaluate with accuracy, precision, recall, F1, ROC AUC
- Show feature importance
- Track with MLflow
```

### 🤖 Genie generaría:

```python
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.sklearn

# Preparar datos
X = df_encoded.drop('churn', axis=1)
y = df_encoded['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# MLflow tracking
with mlflow.start_run(run_name="churn_rf_model"):
    # Entrenar
    model = RandomForestClassifier(
        n_estimators=100, 
        max_depth=10, 
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    # Evaluar
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Métricas
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"\nROC AUC: {roc_auc:.4f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    print("\nTop 5 Features:")
    print(feature_importance.head())
    
    # Log en MLflow
    mlflow.log_params({"n_estimators": 100, "max_depth": 10})
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(model, "random_forest_model")
```

### ✅ Beneficio

**Setup completo de entrenamiento en segundos**, incluyendo:
* Stratified split para clases desbalanceadas
* Múltiples métricas
* Feature importance automático
* MLflow tracking integrado

## Ejemplo 4: Debugging de Errores Comunes

### 🐛 Problema:

```python
# Este código da error
model.fit(X_train_scaled, y_train)
# ValueError: Input contains NaN
```

### 💬 Prompt:

```
My model training fails with "Input contains NaN". 
How do I find which columns have missing values and handle them?
```

### 🤖 Genie respondería:

```python
# 1. Detectar valores faltantes
print("Missing values por columna:")
missing = X_train.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal missing: {X_train.isnull().sum().sum()}")

# 2. Visualizar patrón
import plotly.express as px
missing_pct = (X_train.isnull().sum() / len(X_train) * 100).sort_values(ascending=False)
fig = px.bar(x=missing_pct.index, y=missing_pct.values, 
             title="% Missing Values por Feature")
fig.show()

# 3. Estrategias de manejo

# Opción A: Imputar con media/mediana (numéricas)
from sklearn.impute import SimpleImputer
imputer_num = SimpleImputer(strategy='median')
X_train_imputed = imputer_num.fit_transform(X_train.select_dtypes(include=['number']))
X_test_imputed = imputer_num.transform(X_test.select_dtypes(include=['number']))

# Opción B: Imputar con moda (categóricas)
imputer_cat = SimpleImputer(strategy='most_frequent')
X_train_cat = imputer_cat.fit_transform(X_train.select_dtypes(include=['object']))

# Opción C: Eliminar filas (si <5% missing)
if (X_train.isnull().sum().sum() / X_train.size) < 0.05:
    X_train_clean = X_train.dropna()
    y_train_clean = y_train.loc[X_train_clean.index]
    print("Eliminadas filas con NaN (<5% del total)")

# Recomendación automática
if missing.max() / len(X_train) > 0.5:
    print("⚠️  Columnas con >50% missing: considerar eliminar")
else:
    print("✅ Recomendación: Imputación (Opción A/B)")
```

### ✅ Beneficio

**Diagnostica Y soluciona** en un solo paso:
* Identifica problema
* Visualiza impacto
* Ofrece múltiples soluciones
* Recomienda la mejor opción

## Ejemplo 5: Optimizar Performance del Modelo

### 💬 Prompt:

```
My churn model has 78% accuracy but only 35% recall. 
I need to catch more churners (improve recall) even if accuracy drops slightly.
Suggest 3 approaches with code.
```

### 🤖 Genie sugeriría:

```python
# Enfoque 1: Ajustar threshold de clasificación
print("=== ENFOQUE 1: Threshold Tuning ===")
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Probar diferentes thresholds
from sklearn.metrics import recall_score, precision_score
for threshold in [0.3, 0.4, 0.5, 0.6]:
    y_pred_custom = (y_pred_proba > threshold).astype(int)
    recall = recall_score(y_test, y_pred_custom)
    precision = precision_score(y_test, y_pred_custom)
    print(f"Threshold {threshold}: Recall={recall:.2%}, Precision={precision:.2%}")

# Enfoque 2: Balancear clases con class_weight
print("\n=== ENFOQUE 2: Class Weighting ===")
model_balanced = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',  # Penaliza más clase minoritaria
    random_state=42
)
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)
print(f"Recall: {recall_score(y_test, y_pred_balanced):.2%}")

# Enfoque 3: SMOTE (oversampling sintético)
print("\n=== ENFOQUE 3: SMOTE ===")
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

model_smote = RandomForestClassifier(n_estimators=100, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = model_smote.predict(X_test)
print(f"Recall: {recall_score(y_test, y_pred_smote):.2%}")

# Comparación
print("\n=== COMPARACIÓN ===")
comparison = pd.DataFrame({
    'Método': ['Original', 'Threshold=0.3', 'Class Weight', 'SMOTE'],
    'Recall': [
        recall_score(y_test, y_pred),
        recall_score(y_test, (y_pred_proba > 0.3).astype(int)),
        recall_score(y_test, y_pred_balanced),
        recall_score(y_test, y_pred_smote)
    ]
})
print(comparison.sort_values('Recall', ascending=False))
```

### ✅ Beneficio

**Múltiples estrategias probadas** automáticamente:
* Threshold tuning (más rápido)
* Class weighting (sin aumentar datos)
* SMOTE (balance sintético)
* Comparación side-by-side

## 📚 Mejores Prácticas con Genie Code

### ✅ Cómo Hacer Buenos Prompts

#### 1. **Ser Específico**

❌ **Malo**: "Train a model"  
✅ **Bueno**: "Train a Random Forest with 100 trees for binary classification on imbalanced data"

#### 2. **Dar Contexto**

❌ **Malo**: "Fix this error"  
✅ **Bueno**: "My RandomForest gives ValueError: Input contains NaN during fit(). Dataset has 15% missing values in 'income' column"

#### 3. **Especificar Formato de Salida**

✅ "Generate code using sklearn"  
✅ "Explain in 3 bullet points"  
✅ "Create a Plotly interactive chart"

#### 4. **Solicitar Mejores Prácticas**

✅ "Add error handling and input validation"  
✅ "Include docstrings and type hints"  
✅ "Follow PEP 8 style guide"

---

## 🚀 Casos de Uso Principales

| Tarea | Prompt Ejemplo | Tiempo Ahorrado |
|-------|----------------|------------------|
| **Setup datos** | "Create train/test split stratified by target" | 10-15 min |
| **Feature eng** | "Generate polynomial features degree 2" | 15-20 min |
| **Entrenamiento** | "Train XGBoost with early stopping" | 20-30 min |
| **Evaluación** | "Create confusion matrix and ROC curve" | 10-15 min |
| **Debugging** | "Why is my model overfitting?" | 30-60 min |
| **Optimización** | "Tune hyperparameters with GridSearch" | 20-40 min |
| **Visualización** | "Plot feature importance as horizontal bar" | 5-10 min |

**Total ahorro promedio: 2-3 horas por pipeline**

---

## 📈 Comparación: Manual vs Genie

### Desarrollar Pipeline Completo de Churn

| Etapa | Manual | Con Genie | Ahorro |
|-------|--------|-----------|--------|
| **Setup datos** | 20 min | 3 min | 85% |
| **EDA** | 30 min | 5 min | 83% |
| **Feature eng** | 45 min | 10 min | 78% |
| **Entrenamiento** | 40 min | 8 min | 80% |
| **Evaluación** | 25 min | 5 min | 80% |
| **Optimización** | 60 min | 15 min | 75% |
| **Documentación** | 30 min | 5 min | 83% |
| **TOTAL** | **4h 10min** | **51 min** | **80%** |

🎯 **Genie reduce desarrollo en ~80%**

---

## ⚠️ Limitaciones y Cuándo NO Usar Genie

### Limitaciones

1. **Creatividad limitada**: No innova en arquitecturas nuevas
2. **Domain knowledge**: No entiende lógica de negocio específica
3. **Depuración profunda**: Errores complejos requieren expertise humano
4. **Optimización extrema**: Performance crítico necesita tuning manual

### Cuándo NO depender solo de Genie

❌ Arquitecturas custom (e.g., Graph Neural Networks)  
❌ Lógica de negocio compleja y domain-specific  
❌ Performance crítico (sistemas de trading, medicina)  
❌ Research de nuevas técnicas  
❌ Auditoría y compliance regulatorio  

---

## 📝 Conclusiones

### 🎯 Key Takeaways

1. **Genie Code acelera desarrollo de ML en ~80%**
   - Genera código boilerplate instantáneamente
   - Aplica mejores prácticas automáticamente
   - Debuggea errores con contexto

2. **Mejores prompts = Mejores resultados**
   - Ser específico y dar contexto
   - Especificar formato de salida
   - Iterar y refinar

3. **Casos de uso principales**
   - Setup rápido de pipelines
   - Feature engineering instantáneo
   - Debugging guiado
   - Optimización asistida
   - Aprendizaje de nuevas técnicas

4. **Complementa, no reemplaza**
   - Expertise humano sigue siendo crucial
   - Domain knowledge irremplazable
   - Mejor uso: Genie para velocidad + Humano para estrategia

---

### 🚀 Próximos Pasos

**En el siguiente notebook** (`MLflow_Experiment_Tracking.ipynb`):
* Tracking avanzado de experimentos
* Model Registry y versionado
* Comparación de runs
* Despliegue de modelos

---

### 💡 Reflexión Final

> **"Genie Code no reemplaza a los Data Scientists; los hace 10x más productivos, liberándolos para enfocarse en lo que realmente importa: resolver problemas de negocio."**

**Fórmula del éxito:**

```
🚀 ML de Alto Impacto = Genie (velocidad) + 
                          Humano (estrategia) + 
                          Iteración (mejora continua)
```

**¡Listo para combinar Genie con MLflow tracking en el siguiente notebook!** 🎉